# Skin Condition Classifier — Phase 3: Model Training

This notebook trains a MobileNetV2 CNN using transfer learning on the 15-class skin condition dataset prepared in Phase 2.

**Important:** Run this in the same Colab session as Phase 2 so the processed data is still available. If you started a new session, go back and run Phase 2 first.

**Run each cell from top to bottom. Do not skip any cells.**

## Cell 1 — Check GPU is available
Training on CPU would take hours. This confirms we have a GPU ready.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU and restart.')

## Cell 2 — Import libraries

In [ ]:
import os
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print('Libraries loaded.')

## Cell 3 — Verify split data exists
Confirms the Phase 2 data is still available in this session.

In [ ]:
SPLIT_DIR = Path('/content/data/split')

for split in ['train', 'val', 'test']:
    split_path = SPLIT_DIR / split
    if not split_path.exists():
        print(f'ERROR: {split} folder not found. Please re-run Phase 2 notebook first.')
    else:
        classes = [f for f in split_path.iterdir() if f.is_dir()]
        total = sum(len(list(c.glob('*.jpg'))) for c in classes)
        print(f'{split}: {len(classes)} classes, {total} images — OK')

## Cell 4 — Define data transforms
Transforms are the image preprocessing steps applied before each image is fed to the model.

- **Train transforms** include augmentation (random flips, rotations, color jitter) to artificially expand the dataset and help smaller classes
- **Val/test transforms** only normalize — no augmentation, so evaluation is consistent
- The mean and std values are ImageNet standards that MobileNetV2 was pretrained on

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}

print('Transforms defined.')
print('Train augmentations: horizontal flip, rotation, color jitter, affine translate')
print('Val/Test: resize + normalize only')

## Cell 5 — Load datasets and create DataLoaders
PyTorch's ImageFolder automatically assigns labels based on subfolder names — the folder structure we built in Phase 2 pays off here.

In [ ]:
BATCH_SIZE = 32

image_datasets = {
    split: datasets.ImageFolder(
        root=str(SPLIT_DIR / split),
        transform=data_transforms[split]
    )
    for split in ['train', 'val', 'test']
}

dataloaders = {
    split: DataLoader(
        image_datasets[split],
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        num_workers=2,
        pin_memory=True
    )
    for split in ['train', 'val', 'test']
}

class_names = image_datasets['train'].classes
NUM_CLASSES = len(class_names)

print(f'Number of classes: {NUM_CLASSES}')
print(f'Classes: {class_names}')
print(f'Train batches: {len(dataloaders["train"])}')
print(f'Val batches:   {len(dataloaders["val"])}')
print(f'Test batches:  {len(dataloaders["test"])}')

## Cell 6 — Load pretrained MobileNetV2 and modify for our task
This is the transfer learning step. We load MobileNetV2 pretrained on ImageNet, freeze all its layers, and replace only the final classification layer with a new one that outputs 15 classes instead of 1000.

In [ ]:
model = models.mobilenet_v2(weights='IMAGENET1K_V1')

# Freeze all base layers — we keep ImageNet knowledge intact
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier head with our own 15-class version
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(512, NUM_CLASSES)
)

model = model.to(device)

# Count trainable vs frozen parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total:,}')
print(f'Trainable parameters: {trainable:,} ({100*trainable/total:.1f}%)')
print(f'Frozen parameters:    {total-trainable:,}')
print('\nModel ready. Only the new classifier head will be trained in Phase A.')

## Cell 7 — Define loss function and optimizer
We use class weights to help the model pay more attention to underrepresented classes like urticaria (265 images) vs eczema (1000 images).

In [ ]:
# Compute class weights inversely proportional to class frequency
class_counts = []
for cls in class_names:
    cls_path = SPLIT_DIR / 'train' / cls
    class_counts.append(len(list(cls_path.glob('*.jpg'))))

class_counts = torch.tensor(class_counts, dtype=torch.float)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Only optimize the classifier head parameters
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

# Learning rate scheduler — reduces LR when val loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

print('Class weights (higher = more attention):')
for cls, w in zip(class_names, class_weights.cpu()):
    print(f'  {cls}: {w:.3f}')

## Cell 8 — Training loop (Phase A: train classifier head only)
We train for 10 epochs with the base model frozen. This teaches the new classifier head to work with MobileNetV2's existing features.

You will see train loss, train accuracy, val loss, and val accuracy printed after each epoch. Watch the val accuracy — that is the real measure of how well the model generalizes.

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs, phase_name):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}', end=' | ')
        start = time.time()

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc  = running_corrects.double() / len(image_datasets[phase])
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'train':
                print(f'Train loss: {epoch_loss:.4f} acc: {epoch_acc:.4f}', end=' | ')
            else:
                elapsed = time.time() - start
                print(f'Val loss: {epoch_loss:.4f} acc: {epoch_acc:.4f} ({elapsed:.0f}s)')
                scheduler.step(epoch_loss)
                if epoch_acc > best_val_acc:
                    best_val_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())

    print(f'\nBest val accuracy ({phase_name}): {best_val_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model, history

print('Starting Phase A training (classifier head only, 10 epochs)...')
print('This will take approximately 15-20 minutes.')
print('-' * 70)
model, history_a = train_model(model, criterion, optimizer, scheduler,
                                num_epochs=10, phase_name='Phase A')

## Cell 9 — Plot Phase A training curves
Visualize how loss and accuracy changed across epochs. A healthy training run shows loss decreasing and accuracy increasing for both train and val.

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history['train_loss'], label='Train', marker='o')
    ax1.plot(history['val_loss'],   label='Val',   marker='o')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['train_acc'], label='Train', marker='o')
    ax2.plot(history['val_acc'],   label='Val',   marker='o')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_history(history_a, 'Phase A — Classifier head training')

## Cell 10 — Phase B: Fine-tuning (unfreeze top layers)
Now we unfreeze the top layers of MobileNetV2 and train with a very low learning rate. This lets the model adapt its deeper features to skin condition images specifically, usually boosting accuracy by 3-8%.

In [ ]:
# Unfreeze the last 3 feature blocks of MobileNetV2
for i, layer in enumerate(model.features):
    if i >= 15:  # Unfreeze blocks 15, 16, 17, 18
        for param in layer.parameters():
            param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters after unfreezing: {trainable:,}')

# Much lower learning rate for fine-tuning to avoid destroying pretrained weights
optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='min', factor=0.5, patience=2, verbose=True
)

print('Starting Phase B fine-tuning (top layers unfrozen, 10 epochs)...')
print('This will take approximately 20-25 minutes.')
print('-' * 70)
model, history_b = train_model(model, criterion, optimizer_ft, scheduler_ft,
                                num_epochs=10, phase_name='Phase B')

## Cell 11 — Plot Phase B training curves

In [ ]:
plot_history(history_b, 'Phase B — Fine-tuning top layers')

## Cell 12 — Evaluate on test set
The test set has never been seen during training or validation. This is the true measure of model performance.

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = np.mean(all_preds == all_labels)
print(f'Test accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')
print()
print('Per-class report:')
print(classification_report(all_labels, all_preds, target_names=class_names))

## Cell 13 — Confusion matrix
Shows which classes the model confuses with each other. Bright diagonal = good predictions. Off-diagonal = misclassifications.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Confusion matrix (normalized)')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## Cell 14 — Save the trained model
Saves the model weights and the class names together in one file. This is what the Streamlit app will load in Phase 5.

In [ ]:
os.makedirs('/content/model', exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'num_classes': NUM_CLASSES,
    'architecture': 'mobilenet_v2',
    'test_accuracy': float(test_acc),
}, '/content/model/skin_classifier.pt')

size_mb = os.path.getsize('/content/model/skin_classifier.pt') / 1e6
print(f'Model saved: skin_classifier.pt ({size_mb:.1f} MB)')
print(f'Test accuracy: {test_acc*100:.1f}%')
print('Ready to download.')

## Cell 15 — Download the model
Downloads `skin_classifier.pt` to your computer. Save it into your project's `model/` folder.

**Note:** This file will be around 14MB. It is in .gitignore so it will not be pushed to GitHub — that is intentional.

In [ ]:
from google.colab import files
files.download('/content/model/skin_classifier.pt')
print('Download started. Save to your local model/ folder.')

## Phase 3 complete

Your trained model is saved locally at `model/skin_classifier.pt`.

**What to check before moving to Phase 4:**
- Overall test accuracy should be above 60% — anything above 70% is strong for a 15-class problem
- The confusion matrix diagonal should be mostly bright
- If any class has very low recall (below 0.3), note it — we can discuss whether to drop or merge that class

**Next step:** Phase 4 — evaluation, analysis, and building the Streamlit app.